In [73]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, PowerTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import KFold
from google.colab import files

In [2]:
df = pd.read_csv('carsome_data_super_cleaned.csv')

In [3]:
df

,Year,Model,Total_Price(RM),Price_per_month(RM),Mileage(km),Transmission,Location,Highlight,URL,Brand
0,2017,myvi se 1.5,35400.0,406.0,92189.0,A,"carsome showroom setia spice, pulau pinang 14,...",national daily drive,https://www.carsome.my/buy-car/perodua/myvi/20...,Perodua
1,2023,forester l eyesight 2.0,115800.0,1137.0,14200.0,A,"carsome pj automall, selangor 14,748.1 km\n ...",compact suv,https://www.carsome.my/buy-car/subaru/forester...,Subaru
2,2018,jazz s i-vtec 1.5,52900.0,611.0,75759.0,A,"carsome showroom kuala terengganu, terengganu ...",daily drive,https://www.carsome.my/buy-car/honda/jazz/2018...,Honda
3,2016,cx-3 skyactiv 2.0,55800.0,711.0,59833.0,A,"carsome showroom kota kinabalu, sabah 13,814.5...",compact suv,https://www.carsome.my/buy-car/mazda/cx-3/2016...,Mazda
4,2021,myvi av 1.5,49900.0,460.0,69974.0,A,"carsome showroom johor jaya, johor 14,832.8 km...",national daily drive,https://www.carsome.my/buy-car/perodua/myvi/20...,Perodua
...,...,...,...,...,...,...,...,...,...,...
1660,2017,vios trd sportivo 1.5,53600.0,715.0,36537.0,A,"carsome showroom kota kinabalu, sabah 13,814.5...",daily drive,https://www.carsome.my/buy-car/toyota/vios/201...,Toyota
1661,2020,c-hr 1.8,101800.0,1116.0,40220.0,A,"carsome pj automall, selangor 14,748.1 km\n ...",compact suv,https://www.carsome.my/buy-car/toyota/c-hr/202...,Toyota
1662,2018,3 skyactiv-g high 2.0,69000.0,828.0,97077.0,A,"carsome showroom kempas, johor 14,839.0 km\n ...",upgrade daily drive,https://www.carsome.my/buy-car/mazda/3/2018-ma...,Mazda
1663,2016,vios gx 1.5,50500.0,686.0,144662.0,A,"carsome showroom setia spice, pulau pinang 14,...",daily drive,https://www.carsome.my/buy-car/toyota/vios/201...,Toyota


In [4]:
num_features = ['Year', 'Mileage(km)']
cat_features = ['Brand', 'Model', 'Transmission', 'Location']

In [5]:
X = df.drop(['Total_Price(RM)'], axis=1)
y = df['Total_Price(RM)']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [10]:
y_train = np.log1p(y_train)

In [13]:
num_transformer = Pipeline(steps=[
    ('power', PowerTransformer(method='box-cox')),
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler())
])

cat_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

In [18]:
pipe_RFR = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

pipe_XGB = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42))
])

pipe_LGBM = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMRegressor(random_state=42))
])

pipe_CatB = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostRegressor(verbose=0, random_state=42))
])

In [19]:
pipe_RFR.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [20]:
y_pred_RFR = np.expm1(pipe_RFR.predict(X_test))

In [23]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_RFR))
print("Test RMSE:", rmse)

Test RMSE: 6912.300733666908


In [24]:
pipe_XGB.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('model',
                 XGBReg...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [25]:
y_pred_XGB = np.expm1(pipe_XGB.predict(X_test))

In [26]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_XGB))
print("Test RMSE:", rmse)

Test RMSE: 8254.073781666517


In [27]:
pipe_LGBM.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000224 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 427
[LightGBM] [Info] Number of data points in the train set: 1248, number of used features: 5
[LightGBM] [Info] Start training from score 10.727346


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('model', LGBMRegressor(random_state=42))])

In [28]:
y_pred_LGBM = np.expm1(pipe_LGBM.predict(X_test))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [29]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_LGBM))
print("Test RMSE:", rmse)

Test RMSE: 6474.652684620754


In [30]:
pipe_CatB.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('model',
                 <catboost.core.CatBoostRegressor object at 0x78d535f42a50>)])

In [31]:
y_pred_CatB = np.expm1(pipe_CatB.predict(X_test))

In [32]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_CatB))
print("Test RMSE:", rmse)

Test RMSE: 6961.858176220826


In [33]:
rmse_RFR = np.sqrt(mean_squared_error(y_test, y_pred_RFR))
rmse_XGB = np.sqrt(mean_squared_error(y_test, y_pred_XGB))
rmse_LGBM = np.sqrt(mean_squared_error(y_test, y_pred_LGBM))
rmse_CatB = np.sqrt(mean_squared_error(y_test, y_pred_CatB))
print("Test RMSE RFR:", rmse_RFR)
print("Test RMSE XGB:", rmse_XGB)
print("Test RMSE LGBM:", rmse_LGBM)
print("Test RMSE CatB:", rmse_CatB)

Test RMSE RFR: 6912.300733666908
Test RMSE XGB: 8254.073781666517
Test RMSE LGBM: 6474.652684620754
Test RMSE CatB: 6961.858176220826


In [36]:
estimators = [
    ('rfr', RandomForestRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42)),
    ('lgbm', LGBMRegressor(random_state=42)),
    ('cat', CatBoostRegressor(verbose=0, random_state=42))
]

meta_model = LinearRegression()

In [39]:
stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=True,
    cv=5,
    n_jobs=-1
)

pipe_all = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('stacking', stack_model)
])

In [40]:
pipe_all.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('stacking',
                 Sta...
                                                             max_delta_step=None,
                                                             max_depth=None,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=None,
                                                             n_jobs=None,
                                                             num_parallel_tree=None, ...)),
                                               ('lgbm',
                                                LGBMRegressor(random_state=42)),
                                               ('cat',
                                                <catboost.core.CatBoostRegressor object at 0x78d536fe12b0>)],
                                   final_estimator=LinearRegression(),
                                   n_jobs=-1, passthrough=True))])

In [41]:
y_pred_all = np.expm1(pipe_all.predict(X_test))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [42]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_all))
print("Test RMSE:", rmse)

Test RMSE: 7194.526283577518


In [46]:
estimators = [
    ('rfr', RandomForestRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42)),
    ('lgbm', LGBMRegressor(random_state=42)),
    ('cat', CatBoostRegressor(verbose=0, random_state=42))
]

meta_model = Ridge()

In [47]:
stack_model1 = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=True,
    cv=5,
    n_jobs=-1
)

pipe_all1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('stacking', stack_model1)
])

In [48]:
pipe_all1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('stacking',
                 Sta...
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=None,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=None,
                                                             n_jobs=None,
                                                             num_parallel_tree=None, ...)),
                                               ('lgbm',
                                                LGBMRegressor(random_state=42)),
                                               ('cat',
                                                <catboost.core.CatBoostRegressor object at 0x78d536f435c0>)],
                                   final_estimator=Ridge(), n_jobs=-1,
                                   passthrough=True))])

In [49]:
y_pred_all1 = np.expm1(pipe_all1.predict(X_test))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [50]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_all1))
print("Test RMSE:", rmse)

Test RMSE: 7079.375208855276


In [53]:
estimators = [
    ('rfr', RandomForestRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42)),
    ('lgbm', LGBMRegressor(random_state=42)),
    ('cat', CatBoostRegressor(verbose=0, random_state=42))
]

meta_model = Lasso()

In [54]:
stack_model2 = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=True,
    cv=5,
    n_jobs=-1
)

pipe_all2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('stacking', stack_model2)
])

In [55]:
pipe_all2.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('power',
                                                                   PowerTransformer(method='box-cox')),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Year', 'Mileage(km)']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Brand', 'Model',
                                                   'Transmission',
                                                   'Location'])])),
                ('stacking',
                 Sta...
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=None,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=None,
                                                             n_jobs=None,
                                                             num_parallel_tree=None, ...)),
                                               ('lgbm',
                                                LGBMRegressor(random_state=42)),
                                               ('cat',
                                                <catboost.core.CatBoostRegressor object at 0x78d535f553d0>)],
                                   final_estimator=Lasso(), n_jobs=-1,
                                   passthrough=True))])

In [56]:
y_pred_all2 = np.expm1(pipe_all2.predict(X_test))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [57]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_all2))
print("Test RMSE:", rmse)

Test RMSE: 22905.942302807045


#Tuning

In [64]:
def objective(trial):
    # Param untuk base learners
    rfr_params = {
        'n_estimators': trial.suggest_categorical('rfr__n_estimators', [100, 200, 300]),
        'max_depth': trial.suggest_categorical('rfr__max_depth', [None, 10, 20, 30]),
        'min_samples_split': trial.suggest_categorical('rfr__min_samples_split', [2, 5, 10]),
        'min_samples_leaf': trial.suggest_categorical('rfr__min_samples_leaf', [1, 2, 4]),
        'max_features': trial.suggest_categorical('rfr__max_features', ['sqrt', 'log2'])
    }

    xgb_params = {
        'n_estimators': trial.suggest_categorical('xgb__n_estimators', [100, 300, 500]),
        'learning_rate': trial.suggest_categorical('xgb__learning_rate', [0.01, 0.05, 0.1]),
        'max_depth': trial.suggest_categorical('xgb__max_depth', [3, 5, 7]),
        'subsample': trial.suggest_categorical('xgb__subsample', [0.6, 0.8, 1.0]),
        'colsample_bytree': trial.suggest_categorical('xgb__colsample_bytree', [0.6, 0.8, 1.0]),
        'reg_lambda': trial.suggest_categorical('xgb__reg_lambda', [1, 1.5, 2]),
        'reg_alpha': trial.suggest_categorical('xgb__reg_alpha', [0, 0.5, 1])
    }

    lgbm_params = {
        'n_estimators': trial.suggest_categorical('lgbm__n_estimators', [100, 300, 500]),
        'learning_rate': trial.suggest_categorical('lgbm__learning_rate', [0.01, 0.05, 0.1]),
        'max_depth': trial.suggest_categorical('lgbm__max_depth', [-1, 10, 20]),
        'num_leaves': trial.suggest_categorical('lgbm__num_leaves', [31, 50, 100]),
        'subsample': trial.suggest_categorical('lgbm__subsample', [0.6, 0.8, 1.0]),
        'colsample_bytree': trial.suggest_categorical('lgbm__colsample_bytree', [0.6, 0.8, 1.0]),
        'reg_lambda': trial.suggest_categorical('lgbm__reg_lambda', [0, 1, 2]),
        'reg_alpha': trial.suggest_categorical('lgbm__reg_alpha', [0, 1, 2])
    }

    cat_params = {
        'iterations': trial.suggest_categorical('cat__iterations', [300, 500, 800]),
        'learning_rate': trial.suggest_categorical('cat__learning_rate', [0.01, 0.05, 0.1]),
        'depth': trial.suggest_categorical('cat__depth', [4, 6, 10]),
        'l2_leaf_reg': trial.suggest_categorical('cat__l2_leaf_reg', [1, 3, 5]),
        'border_count': trial.suggest_categorical('cat__border_count', [32, 64, 128]),
        'verbose': 0
    }

    final_estimator_alpha = trial.suggest_categorical('final_alpha', [0.1, 0.5, 1.0])

    # Inisialisasi model
    rfr = RandomForestRegressor(**rfr_params, random_state=42)
    xgb = XGBRegressor(**xgb_params, random_state=42)
    lgbm = LGBMRegressor(**lgbm_params, random_state=42)
    cat = CatBoostRegressor(**cat_params, random_state=42)

    estimators = [
        ('rfr', rfr),
        ('xgb', xgb),
        ('lgbm', lgbm),
        ('cat', cat)
    ]

    stack_model = StackingRegressor(
        estimators=estimators,
        final_estimator=Ridge(alpha=final_estimator_alpha),
        n_jobs=-1
    )

    # Pipeline akhir
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),  # pastikan sudah didefinisikan
        ('model', stack_model)
    ])

    # K-Fold CV
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, scoring='neg_root_mean_squared_error', cv=cv, n_jobs=-1)

    return -np.mean(scores)


In [68]:
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=500, timeout=8100)

print("Best trial:")
print(study.best_trial)

[I 2025-10-11 23:15:07,373] A new study created in memory with name: no-name-b59c5338-8615-4c91-9fad-056cf4de98b7
[I 2025-10-11 23:15:47,160] Trial 0 finished with value: 0.12442055147490777 and parameters: {'rfr__n_estimators': 200, 'rfr__max_depth': None, 'rfr__min_samples_split': 2, 'rfr__min_samples_leaf': 2, 'rfr__max_features': 'sqrt', 'xgb__n_estimators': 500, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 7, 'xgb__subsample': 0.8, 'xgb__colsample_bytree': 0.8, 'xgb__reg_lambda': 1, 'xgb__reg_alpha': 0.5, 'lgbm__n_estimators': 500, 'lgbm__learning_rate': 0.1, 'lgbm__max_depth': 10, 'lgbm__num_leaves': 31, 'lgbm__subsample': 1.0, 'lgbm__colsample_bytree': 0.8, 'lgbm__reg_lambda': 1, 'lgbm__reg_alpha': 2, 'cat__iterations': 800, 'cat__learning_rate': 0.1, 'cat__depth': 6, 'cat__l2_leaf_reg': 1, 'cat__border_count': 64, 'final_alpha': 0.5}. Best is trial 0 with value: 0.12442055147490777.
[I 2025-10-11 23:16:37,942] Trial 1 finished with value: 0.13940818187661813 and parameters: {'r

Best trial:
FrozenTrial(number=308, state=1, values=[0.11189369232857925], datetime_start=datetime.datetime(2025, 10, 12, 1, 24, 15, 525366), datetime_complete=datetime.datetime(2025, 10, 12, 1, 24, 40, 525854), params={'rfr__n_estimators': 300, 'rfr__max_depth': None, 'rfr__min_samples_split': 2, 'rfr__min_samples_leaf': 4, 'rfr__max_features': 'sqrt', 'xgb__n_estimators': 300, 'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__subsample': 1.0, 'xgb__colsample_bytree': 0.6, 'xgb__reg_lambda': 1, 'xgb__reg_alpha': 0, 'lgbm__n_estimators': 300, 'lgbm__learning_rate': 0.1, 'lgbm__max_depth': -1, 'lgbm__num_leaves': 100, 'lgbm__subsample': 0.8, 'lgbm__colsample_bytree': 1.0, 'lgbm__reg_lambda': 0, 'lgbm__reg_alpha': 0, 'cat__iterations': 300, 'cat__learning_rate': 0.01, 'cat__depth': 6, 'cat__l2_leaf_reg': 5, 'cat__border_count': 128, 'final_alpha': 0.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'rfr__n_estimators': CategoricalDistribution(choices=(100, 20

In [69]:
best_params = study.best_trial.params

# Random Forest
rfr_best = RandomForestRegressor(
    n_estimators=best_params['rfr__n_estimators'],
    max_depth=best_params['rfr__max_depth'],
    min_samples_split=best_params['rfr__min_samples_split'],
    min_samples_leaf=best_params['rfr__min_samples_leaf'],
    max_features=best_params['rfr__max_features'],
    random_state=42
)

# XGBoost
xgb_best = XGBRegressor(
    n_estimators=best_params['xgb__n_estimators'],
    learning_rate=best_params['xgb__learning_rate'],
    max_depth=best_params['xgb__max_depth'],
    subsample=best_params['xgb__subsample'],
    colsample_bytree=best_params['xgb__colsample_bytree'],
    reg_lambda=best_params['xgb__reg_lambda'],
    reg_alpha=best_params['xgb__reg_alpha'],
    random_state=42
)

# LightGBM
lgbm_best = LGBMRegressor(
    n_estimators=best_params['lgbm__n_estimators'],
    learning_rate=best_params['lgbm__learning_rate'],
    max_depth=best_params['lgbm__max_depth'],
    num_leaves=best_params['lgbm__num_leaves'],
    subsample=best_params['lgbm__subsample'],
    colsample_bytree=best_params['lgbm__colsample_bytree'],
    reg_lambda=best_params['lgbm__reg_lambda'],
    reg_alpha=best_params['lgbm__reg_alpha'],
    random_state=42
)

# CatBoost
cat_best = CatBoostRegressor(
    iterations=best_params['cat__iterations'],
    learning_rate=best_params['cat__learning_rate'],
    depth=best_params['cat__depth'],
    l2_leaf_reg=best_params['cat__l2_leaf_reg'],
    border_count=best_params['cat__border_count'],
    verbose=0,
    random_state=42
)

# Stacking Regressor
stack_best = StackingRegressor(
    estimators=[
        ('rfr', rfr_best),
        ('xgb', xgb_best),
        ('lgbm', lgbm_best),
        ('cat', cat_best)
    ],
    final_estimator=Ridge(alpha=best_params['final_alpha'])
)

# Final Pipeline
final_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', stack_best)
])

# Fit model
final_pipe_fit = final_pipe.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 427
[LightGBM] [Info] Number of data points in the train set: 1248, number of used features: 5
[LightGBM] [Info] Start training from score 10.727346
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [70]:
tune_pred_final_pipe = np.expm1(final_pipe.predict(X_test))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [71]:
rmse = np.sqrt(mean_squared_error(y_test, tune_pred_final_pipe))
print("Test RMSE:", rmse)

Test RMSE: 6282.362510977591


In [74]:
'''TODO: Silahkan simpan model yang kamu miliki'''
import pickle
# Menyimpan model terbaik dengan pickle
pklname = "best_regression.pkl"

with open(pklname, 'wb') as file:
    pickle.dump(final_pipe_fit, file)

files.download(pklname)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>